# Lab03: Story Ending Language Modeling

**Name:** TODO  
**Student ID:** TODO

Train a causal language model and evaluate the likelihood of sentence 5. Complete the TODOs in `model/layers.py`, `model/transformer.py` and this notebook.

```text
2026_DL_Lab03/
├── dataset/
│   ├── train.jsonl
│   ├── val.jsonl
│   └── test_public.jsonl
├── model/
│   ├── layers.py
│   ├── transformer.py
│   └── tokenizer.json
├── Lab03_report_stuID.docx
└── Lab03.ipynb
```

Train from scratch using `train.jsonl` only. No pretrained weights or external training data. Keep the supplied splits and tokenizer unchanged.

Choose your one final model for submission. The final checkpoint need not have the lowest loss.

See the course presentation for assignment rules.

## 1. Environment

Colab: upload the extracted project folder to Google Drive. Adjust the Drive path in the setup cell.

In [ ]:
# Optional: Google Colab only.
# from google.colab import drive
# drive.mount('/content/drive')

# # Modify this part to the path where your code is located
# %cd /content/drive/MyDrive/2026_DL_Lab03


In [ ]:
# Optional: install missing notebook dependencies in your active environment.
# %pip install torch tokenizers numpy matplotlib


In [ ]:
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer

LAB_ROOT = Path.cwd()
DATA_DIR = LAB_ROOT / 'dataset'
MODEL_DIR = LAB_ROOT / 'model'

import sys
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))
from model.transformer import build_model

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
print('Root:', LAB_ROOT)
print('Device:', DEVICE, '| PyTorch:', torch.__version__)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(DEVICE))

## 2. Dataset and Tokenization

```text
BOS sentence1 SEP sentence2 SEP sentence3 SEP sentence4 SEP sentence5 EOS
```

Maximum full-story length: 128 tokens. Exclude longer stories without truncation. Score fifth-sentence tokens and EOS only.


In [ ]:

MAX_LEN = 128
IGNORE_INDEX = -100
TOKENIZER_PATH = MODEL_DIR / 'tokenizer.json'
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
PAD_ID, BOS_ID, EOS_ID, UNK_ID, SEP_ID = [
    tokenizer.token_to_id(token) for token in ('<pad>', '<bos>', '<eos>', '<unk>', '<sep>')
]
assert (PAD_ID, BOS_ID, EOS_ID, UNK_ID, SEP_ID) == (0, 1, 2, 3, 4)
VOCAB_SIZE = tokenizer.get_vocab_size()
assert VOCAB_SIZE == 8000
print('Vocabulary:', VOCAB_SIZE, '| Maximum full-story length:', MAX_LEN)

In [ ]:
def read_jsonl(path):
    with path.open(encoding='utf-8') as f:
        rows = [json.loads(line) for line in f if line.strip()]
    required = ['storyid'] + [f'sentence{i}' for i in range(1, 6)]
    for row in rows:
        assert all(isinstance(row.get(k), str) and row[k].strip() for k in required)
    assert len({row['storyid'] for row in rows}) == len(rows)
    return rows

train_raw = read_jsonl(DATA_DIR / 'train.jsonl')
val_raw = read_jsonl(DATA_DIR / 'val.jsonl')
assert not ({r['storyid'] for r in train_raw} & {r['storyid'] for r in val_raw})
print(f'Train: {len(train_raw):,} | Validation: {len(val_raw):,}')

sample = train_raw[0]
print('Story ID:', sample['storyid'])
for i in range(1, 6):
    print(f'Sentence {i}:', sample[f'sentence{i}'])


In [ ]:
def encode_example(row):
    sentence_ids = [
        tokenizer.encode(row[f'sentence{i}'], add_special_tokens=False).ids
        for i in range(1, 6)
    ]
    prompt_ids = [BOS_ID]
    for ids in sentence_ids[:4]:
        prompt_ids.extend(ids)
        prompt_ids.append(SEP_ID)
    ending_ids = sentence_ids[4] + [EOS_ID]
    return {
        'storyid': row['storyid'],
        'input_ids': prompt_ids + ending_ids,
        'ending_labels': [IGNORE_INDEX] * len(prompt_ids) + ending_ids,
    }

class ROCStoryDataset(Dataset):
    def __init__(self, rows):
        encoded = [encode_example(row) for row in rows]
        self.examples = [example for example in encoded if len(example['input_ids']) <= MAX_LEN]
        self.excluded_count = len(encoded) - len(self.examples)
        if not self.examples:
            raise ValueError('No eligible complete stories.')

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]

def collate_fn(examples):
    length = max(len(example['input_ids']) for example in examples)
    ids = torch.full((len(examples), length), PAD_ID, dtype=torch.long)
    labels = torch.full_like(ids, IGNORE_INDEX)
    for i, example in enumerate(examples):
        n = len(example['input_ids'])
        ids[i, :n] = torch.tensor(example['input_ids'], dtype=torch.long)
        labels[i, :n] = torch.tensor(example['ending_labels'], dtype=torch.long)
    return {'input_ids': ids, 'ending_labels': labels}

train_set = ROCStoryDataset(train_raw)
val_set = ROCStoryDataset(val_raw)
for name, dataset in [('train', train_set), ('validation', val_set)]:
    print(f'{name}: {len(dataset):,} eligible, {dataset.excluded_count:,} excluded')


### Settings


In [ ]:
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
TRAIN_OBJECTIVE = 'ending'  # 'ending' or 'full'
assert TRAIN_OBJECTIVE in ('ending', 'full')
# Training saves here; results and generation load the same checkpoint.
CKPT_PATH = MODEL_DIR / 'best_model.pth'
FINAL_CKPT_PATH = MODEL_DIR / 'final_model.pth'

model_config = dict(
    vocab_size=VOCAB_SIZE,
    max_len=MAX_LEN,
    d_model=256,
    n_heads=8,
    n_layers=4,
    d_ff=1024,
    dropout=0.1,
    pad_id=PAD_ID,
)
training_config = dict(
    seed=SEED, batch_size=BATCH_SIZE, epochs=EPOCHS,
    learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    train_objective=TRAIN_OBJECTIVE, optimizer='AdamW', scheduler=None,
)

def make_loaders(seed):
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_set, batch_size=BATCH_SIZE, shuffle=True, generator=generator,
        collate_fn=collate_fn, num_workers=0, pin_memory=DEVICE.type == 'cuda',
    )
    val_loader = DataLoader(
        val_set, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn, num_workers=0, pin_memory=DEVICE.type == 'cuda',
    )
    return train_loader, val_loader

train_loader, val_loader = make_loaders(SEED)
batch = next(iter(val_loader))
print({key: tuple(value.shape) for key, value in batch.items()})

## 3. Model

Implement the model computation yourself without pretrained models or built-in Transformer/attention modules.


In [ ]:
set_seed(SEED)
model = build_model(model_config).to(DEVICE).float()
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'Total parameters: {n_params:,}')


## 4. Loss and Evaluation

Keep the supplied length rule and evaluation function unchanged.

In [ ]:
def batch_loss_sum(model, batch, objective='ending'):
    ids = batch['input_ids'].to(DEVICE)
    labels = batch['ending_labels'].to(DEVICE)
    logits = model(ids[:, :-1])
    if logits.shape != (ids.size(0), ids.size(1) - 1, VOCAB_SIZE):
        raise ValueError('Incorrect model output shape.')
    if objective == 'ending':
        targets, ignore = labels[:, 1:], IGNORE_INDEX
    elif objective == 'full':
        targets, ignore = ids[:, 1:], PAD_ID
    else:
        raise ValueError(objective)
    count = targets.ne(ignore).sum()
    loss_sum = F.cross_entropy(
        logits.float().reshape(-1, VOCAB_SIZE), targets.reshape(-1),
        ignore_index=ignore, reduction='sum',
    )
    return loss_sum, count

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, total_tokens, stories = 0.0, 0, 0
    for batch in loader:
        loss_sum, count = batch_loss_sum(model, batch, objective='ending')
        total_loss += loss_sum.item()
        total_tokens += count.item()
        stories += batch['input_ids'].size(0)
    if total_tokens == 0:
        raise ValueError('No ending targets to score.')
    loss = total_loss / total_tokens
    if not math.isfinite(loss):
        raise ValueError('Non-finite evaluation loss.')
    return dict(
        ending_loss=loss,
        ending_ppl=math.exp(loss) if loss < 700 else float('inf'),
        ending_tokens=total_tokens,
        stories=stories,
    )


## 5. Training

Complete the optimization TODO in `train_epoch`.


In [ ]:
def train_epoch(model, loader, optimizer, scheduler=None):
    model.train()
    total_loss, total_tokens = 0.0, 0
    for batch in loader:
        optimizer.zero_grad(set_to_none=True)
        loss_sum, count = batch_loss_sum(model, batch, objective=TRAIN_OBJECTIVE)
        loss = loss_sum / count
        if not torch.isfinite(loss):
            raise ValueError('Non-finite training loss.')

        # Backpropagate and update parameters (clip gradients for stability).
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if scheduler is not None:
            scheduler.step()
        total_loss += loss_sum.item()
        total_tokens += count.item()
    return total_loss / total_tokens


### Train

The default example saves the lowest-validation-loss checkpoint to `model/best_model.pth`.
You may change the training pipeline and choose a different checkpoint.
Each run can overwrite its training checkpoint, so preserve experiments you need.

In [ ]:
set_seed(SEED)
train_loader, val_loader = make_loaders(SEED)
model = build_model(model_config).to(DEVICE).float()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = None
history = []
best_val_loss = float('inf')
start_time = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    train_loss = float(train_epoch(model, train_loader, optimizer, scheduler))
    val = evaluate(model, val_loader)
    record = dict(
        epoch=epoch, train_loss=train_loss, val_ending_loss=val['ending_loss'],
        val_ending_ppl=val['ending_ppl'], elapsed_seconds=time.perf_counter() - start_time,
    )
    history.append(record)
    if val['ending_loss'] < best_val_loss:
        best_val_loss = val['ending_loss']
        torch.save(dict(
            model_state_dict=model.state_dict(), config=dict(model_config),
            training_config=dict(training_config), epoch=epoch, val_metrics=val,
        ), CKPT_PATH)
    print(f"Epoch {epoch:02d}/{EPOCHS} | train loss {train_loss:.4f} | "
          f"val loss {val['ending_loss']:.4f} | val PPL {val['ending_ppl']:.3f}")

training_seconds = time.perf_counter() - start_time
print(f'Best validation loss: {best_val_loss:.4f}')
print(f'Training time: {training_seconds:.1f} seconds')

### Learning curves


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
epochs = [row['epoch'] for row in history]
for ax, key, title in zip(
    axes,
    ['train_loss', 'val_ending_loss', 'val_ending_ppl'],
    [f'Train loss ({TRAIN_OBJECTIVE})', 'Validation ending loss', 'Validation ending PPL'],
):
    ax.plot(epochs, [row[key] for row in history])
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Experiment Results

Results and generation use `CKPT_PATH` from Settings.
To inspect another checkpoint, set `CKPT_PATH` after training and rerun this section.
Follow `Lab03_report_stuID.docx` for report requirements.


In [ ]:
def load_model(checkpoint_path, device):
    """Provided: return (float32 eval model, checkpoint metadata).

    Required checkpoint fields: model_state_dict and config.
    Training metadata such as epoch is optional. Selection
    need not minimize validation loss. Loading never trains or reads a dataset.
    """
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if not isinstance(checkpoint, dict):
        raise ValueError('Checkpoint must be a dictionary, not a pickled model.')
    config = checkpoint.get('config')
    if not isinstance(config, dict):
        raise ValueError('Checkpoint must include a model config dictionary.')
    for key, value in [('vocab_size', VOCAB_SIZE), ('max_len', MAX_LEN), ('pad_id', PAD_ID)]:
        if config.get(key) != value:
            raise ValueError(f'Checkpoint config must have {key}={value}.')
    restored = build_model(config).to(device).float()
    restored.load_state_dict(checkpoint['model_state_dict'], strict=True)
    restored.eval()
    return restored, checkpoint


model, checkpoint = load_model(CKPT_PATH, DEVICE)
val_report = evaluate(model, val_loader)
print('Selected checkpoint:', CKPT_PATH)
print('Model settings:', checkpoint['config'])
print('Training settings:', checkpoint.get('training_config', 'not recorded'))
print('Total parameters:', sum(p.numel() for p in model.parameters()))
print('Selected epoch:', checkpoint.get('epoch', 'not recorded'))
print(f"Validation loss: {val_report['ending_loss']:.4f}")
print(f"Validation PPL: {val_report['ending_ppl']:.3f}")
print(f'Training time for the current run: {training_seconds:.1f} seconds')


### Validation generation examples

Run this cell to show 5 validation stories.
The same seed selects the same eligible stories.
Greedy decoding stops at EOS, 40 new tokens, or the 128-token context limit.
Compare the generated ending with reference sentence 5.
These examples are for inspection.

Uses `CKPT_PATH` (`model/best_model.pth` by default).


In [ ]:
@torch.no_grad()
def generate_ending(model, row, max_new_tokens=40):
    model.eval()
    prompt_ids = [BOS_ID]
    for i in range(1, 5):
        prompt_ids.extend(tokenizer.encode(row[f'sentence{i}'], add_special_tokens=False).ids)
        prompt_ids.append(SEP_ID)
    if len(prompt_ids) >= MAX_LEN:
        raise ValueError('Prompt leaves no room for an ending.')
    ids = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    generated_ids = []
    for _ in range(min(max_new_tokens, MAX_LEN - len(prompt_ids))):
        logits = model(ids)[0, -1].clone()
        logits[[PAD_ID, BOS_ID, SEP_ID]] = float('-inf')
        next_id = int(logits.argmax().item())
        generated_ids.append(next_id)
        if next_id == EOS_ID:
            break
        ids = torch.cat([ids, ids.new_tensor([[next_id]])], dim=1)
    return generated_ids


generation_model, generation_checkpoint = load_model(CKPT_PATH, DEVICE)
eligible_val_ids = {example['storyid'] for example in val_set.examples}
generation_examples = random.Random(SEED).sample(
    [row for row in val_raw if row['storyid'] in eligible_val_ids], k=5,
)
generation_results = []
print('Generation checkpoint:', CKPT_PATH)
print('Generation epoch:', generation_checkpoint.get('epoch', 'not recorded'))
for index, row in enumerate(generation_examples, 1):
    generated_ids = generate_ending(generation_model, row)
    ending = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    generation_results.append(dict(storyid=row['storyid'], token_ids=generated_ids, ending=ending))
    print(f"\nExample {index} | Story ID: {row['storyid']}")
    for i in range(1, 5):
        print(f"Context {i}: {row[f'sentence{i}']}")
    print('Generated ending:', ending or '[empty ending]')
    print('Reference ending:', row['sentence5'])


## 7. Final Public Test

Manually copy or rename your chosen checkpoint to `model/final_model.pth` before running this cell.
The fixed evaluation function computes token-weighted ending loss/PPL on reference sentence 5 plus EOS.

In [ ]:
RUN_PUBLIC_TEST = False

if RUN_PUBLIC_TEST:
    # Load the submitted file, never the current training model or CKPT_PATH.
    model, checkpoint = load_model(FINAL_CKPT_PATH, DEVICE)
    test_raw = read_jsonl(DATA_DIR / 'test_public.jsonl')
    test_ids = {row['storyid'] for row in test_raw}
    assert not test_ids.intersection(row['storyid'] for row in train_raw + val_raw)
    test_set = ROCStoryDataset(test_raw)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
    # Course-provided teacher-forced evaluation on sentence 5 plus EOS.
    test_report = evaluate(model, test_loader)
    test_report.update(
        excluded_stories=test_set.excluded_count,
        selected_epoch=checkpoint.get('epoch'),
        checkpoint='model/final_model.pth',
        model_config=dict(checkpoint['config']),
        model_parameters=sum(p.numel() for p in model.parameters()),
    )
    print('Final checkpoint:', FINAL_CKPT_PATH)
    print('Public test:', test_report)


else:
    print('Final public-test evaluation is disabled.')

## 8. Submission

Save the completed notebook with outputs. Submit one chosen final model. Its weights, public-test results and report must agree.

```text
Lab03_<student_id>.zip/
└── Lab03_<student_id>/
    ├── Lab03.ipynb
    ├── Lab03_report_<student_id>.pdf
    └── model/
        ├── layers.py
        ├── transformer.py
        ├── tokenizer.json
        └── final_model.pth
```

The final checkpoint dictionary must contain `model_state_dict` and `config`. Keep training metadata for your report.
Place the completed report PDF at the folder root as shown above.
Compress the entire folder and upload only `Lab03_<student_id>.zip` to E3.
